<a href="https://colab.research.google.com/github/scottn66/255-final-project/blob/wp7-head-skip/notebooks/10_fullres_head_skip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full resolution: the head's spectral skip, on the fused model and the control

Notebook 09 trained the fused transformer and the collapsed control at native 256x256. Both were near zero on the two thin classes at their best epoch (`video_tape` 0.005 and 0.023, `filament` 0.000 and 0.000; `docs/RESULTS.md`). This notebook trains the same two models again with `--head-skip`: the head also gets a 1x1 conv of each pixel's own spectrum at full resolution (224 to 16 channels). Everything else is notebook 09's setting: `FULL_EPOCHS` epochs (60 by default), batch 8, constant learning rate, `--standardize`, one seed. **Runtime → Change runtime type → GPU, then Runtime → Run all.**

| Tag | Model | Flags | Parameters |
|---|---|---|---|
| `tf_fused_full_skip` | BandTransformer 2+6, band tokens, head skip | `--model transformer --lr 3e-4 --head-skip` | 492,599 |
| `tf_collapsed_full_skip` | BandTransformer 0+8, bands mixed first (the control), head skip | `--model transformer --tokenizer collapsed --depth-spectral 0 --depth-spatial 8 --lr 3e-4 --head-skip` | 495,303 |
| `tf_fused_full` | reference: notebook 09's run, no skip | not run here | 488,487 |
| `tf_collapsed_full` | reference: notebook 09's run, no skip | not run here | 491,191 |

`RUN_TAGS` defaults to the fused model with the skip, then the control with the skip. The two reference tags are never launched and never changed. If their folders are in `RUNS` (notebook 09 wrote them there), the table reads them and prints skip minus no skip. If not, the table says so and leaves those lines out.

| Step | What | Stops when |
|---|---|---|
| Setup | clone `BRANCH`, install, mount Drive | no GPU, the clone fails, or the branch does not take the flags |
| Data | reuse `SW_ROOT`, or copy the zip from Drive, or `scripts/get_data.sh` | md5 or split counts wrong |
| Smoke | two steps of each model at full resolution | a parameter line, the tokenizer line, the head-skip line or the resolution is wrong |
| Launch | background `python -m sw.gates run`, one folder per tag | a tag's folder was started with other flags, or a finished tag ran fewer epochs than `FULL_EPOCHS` and `EXTEND` is off |
| Monitor | progress every `POLL_SECONDS` | a run died or is stuck, or its log shows the wrong images, resolution or model |
| Table | best val FG mIoU, per-class IoU, skip minus no skip, fused minus control, the skip probe | |

Checkpoints, `metrics.jsonl` and `train.log` go to Drive (`RUNS/<tag>/`). The cubes stay on `/content`. If the session dies, open the notebook and Run all again: finished tags are skipped and the others resume from `last.pt`.

**The learning rate is constant on purpose.** A constant-rate run can be trained on to more epochs later: raise `FULL_EPOCHS` and Run all again. See the Settings note.

**Cost (an estimate).** Notebook 09 measured 63.4 s per epoch for the fused model and 63.3 s for the control, without the skip, at batch 8 on an L4: about 1 hour per run for 60 epochs, plus the data download. With the skip the time has not been measured.

**Read the result with these limits.**

* One seed, `FULL_EPOCHS` epochs (60 by default), validation only, constant rate. The protocol in `docs/RECIPE.md` is up to 200 epochs, three seeds, and a warmup-then-cosine schedule. This is a screen, not the protocol run.
* The skip was added after notebook 09's validation numbers showed the thin classes near zero. It was chosen on this validation set, so a gain here is not an independent test of it.
* The skip is a path from a pixel's spectrum to its logits that does not pass through the transformer. A gain could come from the skip alone. The last part of the table runs `sw.skip_probe` on each finished run: it scores `best.pt` as trained, with the skip zeroed, and with the token path zeroed. A gain on `video_tape` that stays when the token path is zeroed means the skip alone does it. A low row with the token path zeroed does not show that the transformer is needed, and the row with the skip zeroed is not a run trained without the skip: do not compare it with the reference rows.
* Skip minus no skip compares two separate single-seed runs, and more than the skip differs between them. The head's last two layers (`fuse` and the classifier) start from different random weights, because the skip changes the shape of `fuse`. The no-skip run is from an earlier session (commit 318358e; `src/` has not changed since, apart from the skip). Run-to-run spread is not measured. Both no-skip runs were still rising at epoch 60. A difference of a few points in FG mIoU is not evidence of an effect. A thin class moving up from near zero is the thing to look for.
* The skip is the same on both models (4,112 more parameters each), so fused minus control still compares the two encoders. But with the skip both heads read each pixel's 224 bands directly through a 1x1 conv, which is what the control's tokenizer does. On a class where the tokens-zeroed row is high, fused minus control says little about band tokens. Do not read the change in the gap between the with-skip and without-skip lines as an effect: it is four runs, one seed each.

AI-assisted notebook; Scott reviews every line.

## Settings

Edit here, or set the environment variables (the local test does that). `FULL_EPOCHS` is set once and passed once, inside `SHARED`. There is no `--downsample` flag: `sw.train` then trains at native size and logs `downsample 1`. Each tag computes its own band statistics into its own folder. No tag takes `--stats-path`. The flags of a run tag are its reference tag's flags plus `--head-skip`, and nothing else.

**Training longer.** With `--no-cosine` the learning rate is the same in every epoch, and `last.pt` holds the weights, the optimizer state and the best score. So a run that finished at 60 epochs can be resumed to 100: the weights, the optimizer, the best score and the metrics log carry on, and the rate does not change. Set `FULL_EPOCHS` higher and Run all again. With `EXTEND = True` the Launch cell reopens every finished tag that has fewer epochs and resumes it. One thing does not carry on: the random stream (shuffle order, flips) starts again from the seed at each resume.

A cosine run cannot be extended this way. Its rate falls towards zero as the planned last epoch nears, and on a resume past that point it climbs back up.

Do not point `FULL_RUNS_DIR` at a folder used for a local test or for other data. A tag that resumes reuses the `band_stats.pt` already in its folder and does not compute it again.

In [ ]:
import os
from pathlib import Path

BRANCH = os.environ.get("SW_BRANCH", "wp7-head-skip")  # the branch that has --head-skip
REPO_URL = "https://github.com/scottn66/255-final-project.git"
RUNS = Path(os.environ.get("FULL_RUNS_DIR", "/content/drive/MyDrive/sw_runs/fullres"))  # Drive: one folder per tag; notebook 09's folder
WORK = Path(os.environ.get("FULL_WORK", "/content/sw_fullres_skip"))  # local disk: the smoke runs
DATA_DIR = Path(os.environ.get("FULL_DATA", "/content/data"))  # local disk: the cubes (never train from Drive)
FULL_ZIP = os.environ.get("FULL_ZIP", "")  # optional: a copy of spectralwaste_segmentation.zip on Drive
DELETE_ZIP = True  # free 21 GB of local disk once the cubes are unpacked and counted
DEVICE = os.environ.get("FULL_DEVICE", "cuda")
NUM_WORKERS = int(os.environ.get("FULL_WORKERS", "2"))
POLL_SECONDS = int(os.environ.get("FULL_POLL", "300"))  # how often the monitor prints
# A live run is called stuck when its train.log and metrics.jsonl have not grown for
# max(STALL_FLOOR_SECONDS, STALL_EPOCHS x its median epoch time). The log grows once per epoch.
# The floor covers the band statistics pass and epoch 1. Their time at full resolution is not measured.
STALL_FLOOR_SECONDS = int(os.environ.get("FULL_STALL_FLOOR", "3600"))
STALL_EPOCHS = 5

FULL_EPOCHS = int(os.environ.get("FULL_EPOCHS", "60"))
# True: a tag that finished with fewer than FULL_EPOCHS epochs is reopened and resumed to FULL_EPOCHS.
# False: the Launch cell stops when it finds such a tag.
EXTEND = os.environ.get("FULL_EXTEND", "1") != "0"

# The same for every tag. --epochs is here and nowhere else. No --downsample: native 256x256.
SHARED = [
    "--batch-size", "8",
    "--no-cosine",  # constant lr, so a run can be extended
    "--standardize",
    "--epochs", str(FULL_EPOCHS),
    "--num-workers", str(NUM_WORKERS),
    "--device", DEVICE,
]

# Notebook 09's two models, by its tags. These are the reference tags: never launched and never
# changed here. Their folders in RUNS, if they exist, are read for the table.
BASE = {
    "tf_fused_full": ["--model", "transformer", "--lr", "3e-4"],
    "tf_collapsed_full": [
        "--model", "transformer", "--tokenizer", "collapsed",
        "--depth-spectral", "0", "--depth-spatial", "8", "--lr", "3e-4",
    ],
}
REFERENCE_TAGS = list(BASE)
SKIP_OF = {"tf_fused_full": "tf_fused_full_skip", "tf_collapsed_full": "tf_collapsed_full_skip"}  # reference tag -> its skip tag

# The tags this notebook trains: each reference tag's flags plus --head-skip.
ARMS = {SKIP_OF[tag]: [*flags, "--head-skip"] for tag, flags in BASE.items()}
FULL = {tag: [*flags, *SHARED] for tag, flags in ARMS.items()}
REFERENCE = {tag: [*flags, *SHARED] for tag, flags in BASE.items()}
EXPECTED = {**FULL, **REFERENCE}  # the flags each tag's folder must have been started with
FUSED, CONTROL = "tf_fused_full_skip", "tf_collapsed_full_skip"  # the pair "fused minus control" compares

# Lines every smoke run and every train.log must show for the tag.
PARAM_LINE = {
    "tf_fused_full_skip": "BandTransformer parameters: 492,599 (0.493M)",
    "tf_collapsed_full_skip": "BandTransformer parameters: 495,303 (0.495M)",
    "tf_fused_full": "BandTransformer parameters: 488,487 (0.488M)",
    "tf_collapsed_full": "BandTransformer parameters: 491,191 (0.491M)",
}
TOKENIZER_LINE = {
    "tf_fused_full_skip": "transformer  tokenizer band  blocks 2+6",
    "tf_collapsed_full_skip": "transformer  tokenizer collapsed  blocks 0+8",
    "tf_fused_full": "transformer  tokenizer band  blocks 2+6",
    "tf_collapsed_full": "transformer  tokenizer collapsed  blocks 0+8",
}
# sw.train prints this line only when the head skip is on. A run tag's log must have it; a reference tag's must not.
SKIP_LINE = "head skip  on  1x1 conv 224->16 from the model input, at its resolution"
# What sw.train prints after the image counts when no --downsample flag is given.
RESOLUTION_TEXT = "downsample 1  "

# Which tags this notebook runs, in this order. A reference tag cannot be listed here.
RUN_TAGS = ["tf_fused_full_skip", "tf_collapsed_full_skip"]
if os.environ.get("FULL_TAGS"):
    RUN_TAGS = [tag.strip() for tag in os.environ["FULL_TAGS"].split(",") if tag.strip()]

# True: one run at a time, the next starts when the one before is done.
# GPU memory for two full-resolution runs at once has not been measured. False starts them all together.
ONE_AT_A_TIME = os.environ.get("FULL_ONE_AT_A_TIME", "1") != "0"
WAIT_FOR_RUNS = True  # the monitor waits until every tag is done; interrupting it does not stop a run
# True: the table cell runs sw.skip_probe on each finished run's best.pt (three passes over the validation split).
PROBE = os.environ.get("FULL_PROBE", "1") != "0"
PROBE_TIMEOUT = 1800  # seconds per tag; a probe that takes longer is reported and the table goes on

unknown = [tag for tag in RUN_TAGS if tag not in FULL]
if unknown or not RUN_TAGS or len(set(RUN_TAGS)) != len(RUN_TAGS):
    raise ValueError(f"RUN_TAGS must be distinct names from {list(FULL)}; got {RUN_TAGS}")
for tag, flags in EXPECTED.items():
    if (tag in FULL) != ("--head-skip" in flags):
        raise ValueError(f"{tag}: --head-skip belongs to the run tags and to no reference tag")
    if flags.count("--epochs") != 1:
        raise ValueError(f"{tag}: --epochs must appear once, in SHARED")
    if "--downsample" in flags:
        raise ValueError(f"{tag}: no --downsample here; this notebook trains at native size")
    if "--stats-path" in flags:
        raise ValueError(f"{tag}: no --stats-path here; each tag computes its own band statistics")
    if "--no-cosine" not in flags:
        raise ValueError(f"{tag}: --no-cosine is needed; a cosine run cannot be extended")
for tag, skip_tag in SKIP_OF.items():
    if [flag for flag in FULL[skip_tag] if flag != "--head-skip"] != REFERENCE[tag]:
        raise ValueError(f"{skip_tag} must be {tag}'s flags plus --head-skip and nothing else")

## Setup

On Colab: mount Drive, clone `BRANCH` (or pull it if the clone is already there), install `requirements.txt`, check for a GPU, and check that this branch's `sw.train` takes every tag's flags. A branch without `--head-skip` stops here, before the data step. Locally: use the repo this notebook is in.

After changing `BRANCH`, restart the runtime before Run all. The flag check runs in a new Python process, so it sees the new branch. This notebook's own `sw.gates` import does not: it stays on the code loaded first until the runtime restarts.

In [ ]:
import gc
import json
import re
import shutil
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
T_START = time.time()


def sh(cmd):
    """Run a command, show its output, raise if it fails."""
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True)


def tail(path, n=25):
    return "\n".join(Path(path).read_text(errors="replace").splitlines()[-n:]) if Path(path).is_file() else "(no log)"


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/255-final-project")
    if not REPO.is_dir():
        sh(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)])
    else:
        sh(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH])
        try:
            sh(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"])
        except subprocess.CalledProcessError as err:
            raise RuntimeError(
                f"{REPO} has local changes that block switching to {BRANCH}; "
                "delete the folder and Run all again"
            ) from err
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])
else:
    _env = os.environ.get("REPO_ROOT", "").strip()
    _here = Path.cwd().resolve()
    REPO = Path(_env) if _env else next(
        p for p in [_here, *_here.parents] if (p / "src" / "sw" / "gates.py").is_file()
    )

os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import torch
from sw import gates

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → GPU, then Run all again.")

COMMIT = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO, capture_output=True, text=True
).stdout.strip()
ON_BRANCH = subprocess.run(
    ["git", "branch", "--show-current"], cwd=REPO, capture_output=True, text=True
).stdout.strip()

ENV = {**os.environ, "PYTHONPATH": str(REPO / "src")}

# Every tag's flags must parse on this branch, before the data step. The check runs in a new
# Python process, as the runs do, so it tests the code on disk and not a module imported earlier.
PARSE_CHECK = "import sys; from sw.train import _build_parser; _build_parser().parse_args(sys.argv[1:])"


def flags_error(flags):
    """None when this branch's sw.train takes the flags, else the last line of its error."""
    check = subprocess.run([sys.executable, "-c", PARSE_CHECK, *flags], capture_output=True, text=True, env=ENV, cwd=REPO)
    if check.returncode == 0:
        return None
    return (check.stderr.strip().splitlines() or [f"exit code {check.returncode}"])[-1]


for tag, flags in FULL.items():
    _why = flags_error(flags)
    if _why:
        raise RuntimeError(
            f"{REPO} (branch {ON_BRANCH or BRANCH}, commit {COMMIT}) does not take the flags of {tag}: {_why}\n"
            "This notebook needs --head-skip. Set BRANCH to wp7-head-skip or a later branch."
        )

RUNS.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else DEVICE

print(f"repo   {REPO} @ {COMMIT} (branch {ON_BRANCH or 'detached'})")
print(f"torch  {torch.__version__} on {GPU}")
print(f"tags   {RUN_TAGS}, {FULL_EPOCHS} epochs each, {'one at a time' if ONE_AT_A_TIME else 'all at once'}, extend {'on' if EXTEND else 'off'}")
print(f"runs   {RUNS}\nwork   {WORK}\ndata   {DATA_DIR}")

## Data

Three ways, tried in this order:

1. `SW_ROOT` is set and already holds the full train and val splits. Nothing is downloaded.
2. `FULL_ZIP` points at a copy of the zip on Drive. It is copied to local disk, then checked and unpacked.
3. `scripts/get_data.sh` downloads the zip (resumable), checks its md5, and unzips it.

The counts must be 514 train and 167 val. The monitor checks the same two numbers, and `downsample 1`, in every run's log.

In [ ]:
from sw.data import SPLIT_COUNTS, SpectralWasteDataset


def split_sizes(root):
    return {s: len(SpectralWasteDataset(root, split=s, random_flip=False)) for s in ("train", "val")}


def counts_ok(root):
    try:
        return all(n == SPLIT_COUNTS[s] for s, n in split_sizes(root).items())
    except (FileNotFoundError, ValueError, RuntimeError):
        return False


SW_ROOT = Path(os.environ["SW_ROOT"]) if os.environ.get("SW_ROOT") else DATA_DIR / "spectralwaste_segmentation"
ALLOW_PARTIAL = os.environ.get("FULL_ALLOW_PARTIAL") == "1"  # local test only: a small SW_ROOT
if ALLOW_PARTIAL and (SW_ROOT / "hyper").is_dir():
    print(f"SW_ROOT {SW_ROOT}: partial data allowed (local test; not the official split)")
elif counts_ok(SW_ROOT):
    print(f"SW_ROOT {SW_ROOT}: unpacked, train and val counts match")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(DATA_DIR).free / 2**30
    print(f"local disk free: {free:.0f} GiB (zip 21 GB + cubes 24 GB while unzipping)")
    if free < 50:
        print("WARNING: under 50 GiB free; the unzip may run out of space")
    zip_path = DATA_DIR / "spectralwaste_segmentation.zip"
    if FULL_ZIP and Path(FULL_ZIP).is_file() and not zip_path.is_file():
        sh(["cp", FULL_ZIP, str(zip_path)])
    sh(["bash", str(REPO / "scripts" / "get_data.sh"), str(DATA_DIR)])
    SW_ROOT = DATA_DIR / "spectralwaste_segmentation"
    if not counts_ok(SW_ROOT):
        raise RuntimeError(f"{SW_ROOT}: the train/val counts do not match {SPLIT_COUNTS}")
    if DELETE_ZIP:
        zip_path.unlink(missing_ok=True)
os.environ["SW_ROOT"] = ENV["SW_ROOT"] = str(SW_ROOT)

# What every train.log must show: 514 and 167 (unless this is the local test), at native size.
SIZES = split_sizes(SW_ROOT)
IMAGES_LINE = f"train images {SIZES['train']}  val images {SIZES['val']}  {RESOLUTION_TEXT}"
print(f"every run must log: {IMAGES_LINE.strip()}")

## Smoke: one per model, at full resolution

Two optimizer steps of each of the two models on 4 training images, with `--standardize` and `--head-skip`, before any long run. `--smoke` alone would shrink the images by 4, so this cell passes `--downsample 1`: the smoke takes the same full-resolution path as the real runs. (The real runs get there by having no `--downsample` flag.)

Each smoke must print its parameter line, the head-skip line and `downsample 1`. The fused model must read `492,599`, the control `495,303` and `transformer  tokenizer collapsed  blocks 0+8`. Both must print `head skip  on  1x1 conv 224->16 from the model input, at its resolution`. A wrong or missing line stops the notebook: it would mean this branch does not build the model the table names.

On a GPU the cell then prints the peak GPU memory of one forward and backward pass on a random batch of 8 at 224 x 256 x 256, for each tag in `RUN_TAGS`. It leaves out the optimizer step and the data loader, so a real step can use a little more. A failure there (out of memory, for example) is printed and does not stop the notebook; the memory is handed back before the next tag and before the Launch cell. On CPU that part is skipped. Notebook 09 measured 3.46 GiB (fused) and 3.06 GiB (control) without the skip on an L4. With the skip it has not been measured.

In [ ]:
for tag, flags in ARMS.items():
    smoke = subprocess.run(
        [
            sys.executable, "-m", "sw.train", *flags, "--smoke", "--downsample", "1", "--standardize",
            "--device", DEVICE, "--data-root", str(SW_ROOT),
            "--out-dir", str(WORK / f"smoke_{tag}"),
        ],
        capture_output=True, text=True, env=ENV, cwd=REPO,
    )
    print(f"--- smoke {tag}")
    print(smoke.stdout[-1500:], smoke.stderr[-3000:], sep="\n")
    if smoke.returncode != 0:
        raise RuntimeError(f"smoke {tag} failed: the run raised (traceback above)")
    for line in (PARAM_LINE[tag], TOKENIZER_LINE[tag], SKIP_LINE, "  " + RESOLUTION_TEXT + "crop None"):
        if line and line not in smoke.stdout:
            raise RuntimeError(f"smoke {tag} failed: the output has no line '{line}'")
print("smoke passed for", ", ".join(ARMS))


def one_batch_peak_gib(tag, size=256, bands=224):
    """One forward and backward pass of the tag's model on a random batch of its batch size.

    Returns the peak GPU memory in GiB, or None when DEVICE is not cuda.
    """
    from sw.train import _build_parser, build_model, config_from_args

    args = _build_parser().parse_args(FULL[tag])
    cfg = config_from_args(args)
    on_gpu = DEVICE == "cuda"
    model = build_model(cfg, in_channels=bands).to(DEVICE)
    if on_gpu:
        torch.cuda.reset_peak_memory_stats()
    cubes = torch.randn(cfg.batch_size, bands, size, size, device=DEVICE)
    masks = torch.randint(0, 7, (cfg.batch_size, size, size), device=DEVICE)
    loss = torch.nn.functional.cross_entropy(model(cubes), masks)
    loss.backward()
    return torch.cuda.max_memory_allocated() / 2**30 if on_gpu else None


def measure_memory(tags, measure=one_batch_peak_gib):
    """One printed line per tag. A failure is printed, not raised: the real run's own log is the check that counts."""
    for tag in tags:
        try:
            line = f"{measure(tag):.2f} GiB"
        except Exception as err:
            line = f"could not be measured: {type(err).__name__}: {str(err)[:200]}"
        # Free the memory here, after the except block has ended. Inside it the error still
        # holds the model and the batch, so the cache could not be emptied.
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  {tag:22s} {line}")


if DEVICE == "cuda":
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"\npeak GPU memory, one forward and backward pass, batch 8 at 224 x 256 x 256 (GPU has {total:.1f} GiB)")
    measure_memory(RUN_TAGS)  # only the tags that will run
    print(f"  held by this notebook after the check: {torch.cuda.memory_reserved() / 2**30:.2f} GiB")
else:
    print(f"\npeak GPU memory: skipped (device is {DEVICE})")

## Launch

Each tag is a background `python -m sw.gates run` with its checkpoints and log in `RUNS/<tag>/`. Interrupting a cell does not stop it. Re-running this cell skips finished tags, leaves a live run alone, and resumes a stopped one from its `last.pt`.

With `ONE_AT_A_TIME` this cell starts the first unfinished tag only. The monitor starts the next one when it is done, so keep the monitor running.

First the cell compares the flags stored in every tag's folder (`ARGS.json`) with this notebook's flags for that tag. `--epochs`, `--data-root`, `--device` and `--num-workers` may differ; anything else (batch size, learning rate, schedule, `--standardize`, the model) stops the cell before any file is changed. This covers finished tags too. The reference tags are not in `RUN_TAGS`, so this cell never starts, reopens or extends them.

A finished tag is skipped when it ran `FULL_EPOCHS` epochs or more. When it ran fewer:

* `EXTEND = True`: the cell moves its `DONE.json` aside (to `DONE.after_<n>_epochs.json`), prints a line saying so, and the run resumes from `last.pt` up to `FULL_EPOCHS`.
* `EXTEND = False`: the cell stops and names the tag.

A run that is still training keeps the `--epochs` it was started with. If `FULL_EPOCHS` was raised since, the monitor reopens the run when it finishes and trains it on to `FULL_EPOCHS` (with `EXTEND = False` the monitor stops instead).

In [ ]:
PROCS = {}
STARTED = set()  # tags started, or seen running, since this cell last ran
DEAD = set()  # tags the monitor saw die; re-running this cell clears it


def epochs_done(tag):
    """The last epoch in the tag's metrics.jsonl, or None."""
    try:
        return int(gates.read_metrics(RUNS / tag / "metrics.jsonl")[-1]["epoch"])
    except (FileNotFoundError, ValueError):
        return None


def train_flags(tag):
    """The sw.train flags the tag is launched with (a reference tag: the flags its folder must hold)."""
    return [*EXPECTED[tag], "--data-root", str(SW_ROOT)]


def flags_differ(tag):
    """None when the tag's folder has no ARGS.json or was started with this notebook's flags.

    Otherwise (stored, now): the two flag lists without the flags a resume may change.
    """
    stored = RUNS / tag / gates.ARGS_FILE
    if not stored.is_file():
        return None
    before = gates.run_identity(json.loads(stored.read_text(encoding="utf-8")))
    now = gates.run_identity(train_flags(tag))
    return None if before == now else (before, now)


def check_flags():
    """Raise if a tag's folder, finished or not, was started with other flags. Changes nothing."""
    for tag in RUN_TAGS:
        differ = flags_differ(tag)
        if differ:
            raise RuntimeError(
                f"{tag}: {RUNS / tag} was started with different flags.\n  stored: {differ[0]}\n  now:    {differ[1]}\n"
                "Nothing was changed. Its numbers are not from this notebook's settings. "
                "Put the settings back, or set another runs folder (FULL_RUNS_DIR), or move this folder away."
            )


def short_tags():
    """Finished tags that ran fewer epochs than FULL_EPOCHS, as {tag: epochs run}."""
    return {
        tag: epochs_done(tag) or 0 for tag in RUN_TAGS
        if gates.run_state(RUNS / tag) == "done" and (epochs_done(tag) or 0) < FULL_EPOCHS
    }


def extend_or_refuse():
    """Reopen finished tags that are short of FULL_EPOCHS (EXTEND on), or raise (EXTEND off)."""
    short = short_tags()
    if not short:
        return
    if not EXTEND:
        raise RuntimeError(
            f"finished with fewer than {FULL_EPOCHS} epochs: {short} (tag: epochs run) in {RUNS}.\n"
            f"Set EXTEND = True to train such a tag on to {FULL_EPOCHS}, or lower FULL_EPOCHS, or set another runs folder."
        )
    no_checkpoint = [tag for tag in short if not (RUNS / tag / "last.pt").is_file()]
    if no_checkpoint:
        raise RuntimeError(
            f"{no_checkpoint}: finished with fewer than {FULL_EPOCHS} epochs and there is no last.pt to resume from "
            f"in {RUNS}. Nothing was changed. Put last.pt back, or move the folder away to train from epoch 1."
        )
    for tag, n in short.items():
        done = RUNS / tag / gates.DONE_FILE
        kept = done.with_name(f"DONE.after_{n}_epochs.json")
        done.replace(kept)
        print(
            f"EXTEND {tag}: it finished at {n} epochs and FULL_EPOCHS is {FULL_EPOCHS}. "
            f"Moved {gates.DONE_FILE} to {kept.name}; the run resumes from last.pt at epoch {n + 1}."
        )


def start_due():
    """Start the tags that should be training now. A tag that died is not started again here."""
    for tag in RUN_TAGS:
        if tag in DEAD:
            continue
        if tag not in FULL:  # a reference tag: read only
            raise RuntimeError(f"{tag} is a reference tag; this notebook never launches it")
        state = gates.run_state(RUNS / tag)
        if state == "done":
            continue
        if state != "running" and tag not in STARTED:
            PROCS[tag] = gates.launch(RUNS / tag, train_flags(tag), env=ENV)
        STARTED.add(tag)
        if ONE_AT_A_TIME:
            return


check_flags()
extend_or_refuse()
start_due()

## Monitor

Prints every tag's progress every `POLL_SECONDS`. Interrupt it any time: the runs keep going, and re-running it picks them up.

It stops with an error in three cases.

* **A run died.** Its folder has no `DONE.json` and its process is gone. The end of its `train.log` is printed. The other tags still run to the end; then this cell raises and names the dead ones. Fix the cause, then re-run the Launch cell and this cell: the run resumes from `last.pt`.
* **A run is stuck.** Its process is alive, but its `train.log` and `metrics.jsonl` have not grown for `max(STALL_FLOOR_SECONDS, STALL_EPOCHS x median epoch time)`. The cell raises at once and prints the pid. Nothing is killed. The clock starts when this cell starts.
* **A log shows the wrong run.** Every `train.log` must show the image counts from the Data cell (514 and 167 on the real set) followed by `downsample 1`, the tag's parameter count, the tokenizer line and the head-skip line.

In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:  # no progress bars; the printed lines are enough
    tqdm = None


def log_check(tag):
    """Compare the banner lines in the tag's train.log with what the tag must show.

    Returns (complete, wrong). complete: every expected line was found at
    least once. wrong: banner lines that do not match. A resumed run prints
    the banner again, so every copy is checked.
    """
    log = RUNS / tag / "train.log"
    lines = log.read_text(errors="replace").splitlines() if log.is_file() else []
    images = [line for line in lines if line.startswith("train images ")]
    params = [line for line in lines if re.match(r"\w+ parameters: ", line)]
    tokenizer = [line for line in lines if line.startswith("transformer  tokenizer ")]
    skip = [line for line in lines if line.startswith("head skip")]
    wrong = [line for line in images if not line.startswith(IMAGES_LINE)]
    wrong += [line for line in params if line != PARAM_LINE[tag]]
    want = TOKENIZER_LINE[tag]
    wrong += [line for line in tokenizer if want is None or not line.startswith(want + "  ")]
    # A run tag prints the head-skip line. A reference tag (no skip) must not print one at all.
    wrong += [line for line in skip if tag not in FULL or line != SKIP_LINE]
    complete = bool(images) and bool(params) and (want is None or bool(tokenizer)) and (tag not in FULL or bool(skip))
    return complete, wrong


def status(tag):
    out = RUNS / tag
    state = gates.run_state(out)
    proc = PROCS.get(tag)
    if proc is not None and proc.poll() is None:
        state = "running"
    if state == "new" and tag not in STARTED:
        return state, f"{tag:22s} waiting  not started yet"
    try:
        rows = gates.read_metrics(out / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        return state, f"{tag:22s} {state:8s} no epoch finished yet"
    last = rows[-1]
    best = max(rows, key=lambda r: r.get("val_fg_miou") or -1.0)
    sec = gates.median_epoch_seconds(rows) or 0.0
    left = sec * max(0, FULL_EPOCHS - int(last["epoch"])) / 3600 if state != "done" else 0.0
    return state, (
        f"{tag:22s} {state:8s} epoch {last['epoch']:3d}/{FULL_EPOCHS}  "
        f"val FG {last.get('val_fg_miou') or 0:.3f} (best {best.get('val_fg_miou') or 0:.3f} at {best['epoch']})  "
        f"{sec:.1f} s/epoch  about {left:.1f} h left"
    )


GROWTH = {}  # tag -> (log sizes, when this cell first saw them)


def quiet_seconds(tag):
    """Seconds since this cell last saw the tag's train.log or metrics.jsonl grow."""
    out = RUNS / tag
    sizes = tuple(
        (out / name).stat().st_size if (out / name).is_file() else -1 for name in ("train.log", "metrics.jsonl")
    )
    if tag not in GROWTH or GROWTH[tag][0] != sizes:
        GROWTH[tag] = (sizes, time.time())
    return time.time() - GROWTH[tag][1]


def stall_limit(tag):
    try:
        sec = gates.median_epoch_seconds(gates.read_metrics(RUNS / tag / "metrics.jsonl")) or 0.0
    except (FileNotFoundError, ValueError):
        sec = 0.0
    return max(STALL_FLOOR_SECONDS, STALL_EPOCHS * sec)


BARS = {} if tqdm is None else {
    tag: tqdm(total=FULL_EPOCHS, desc=tag, unit="epoch", position=i) for i, tag in enumerate(RUN_TAGS)
}


def update_bar(tag):
    if tag not in BARS:
        return
    try:
        rows = gates.read_metrics(RUNS / tag / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        return
    best = max(r.get("val_fg_miou") or 0.0 for r in rows)
    BARS[tag].n = int(rows[-1]["epoch"])  # set, not add: safe after a resume
    BARS[tag].set_postfix(best_val=f"{best:.3f}")
    BARS[tag].refresh()


while True:
    if EXTEND and short_tags():
        # A run started at an older, lower FULL_EPOCHS has finished. Reopen it and train it on.
        STARTED.difference_update(short_tags())
        extend_or_refuse()
        start_due()
    report = {tag: status(tag) for tag in RUN_TAGS}
    print(time.strftime("%H:%M:%S"), *[line for _, line in report.values()], sep="\n  ", flush=True)
    for tag in RUN_TAGS:
        update_bar(tag)
        _, wrong = log_check(tag)
        if wrong:
            shown = "\n  ".join(wrong)
            raise RuntimeError(
                f"{tag}: {RUNS / tag / 'train.log'} shows the wrong run:\n  {shown}\n"
                f"expected '{IMAGES_LINE.strip()}', '{PARAM_LINE[tag]}'"
                + (f", '{TOKENIZER_LINE[tag]}'" if TOKENIZER_LINE[tag] else "")
                + f", '{SKIP_LINE}'"
                + f".\nIf it is still training, stop it (its pid is in {RUNS / tag / gates.RUNNING_FILE}). "
                "Then move the folder away and Run all again."
            )
    states = {tag: state for tag, (state, _) in report.items()}
    for tag in RUN_TAGS:
        if tag in STARTED and tag not in DEAD and states[tag] in ("new", "partial"):
            DEAD.add(tag)
            print(f"\n{tag} is not running and not done. End of its log:\n{tail(RUNS / tag / 'train.log')}\n", flush=True)
    for tag in RUN_TAGS:
        if states[tag] == "running":
            quiet, limit = quiet_seconds(tag), stall_limit(tag)
            if quiet > limit:
                print(f"\n{tag} is alive but its log has not grown. End of its log:\n{tail(RUNS / tag / 'train.log')}")
                raise RuntimeError(
                    f"{tag} is stuck: no new line in train.log or metrics.jsonl for {quiet:.0f} s (limit {limit:.0f} s). "
                    f"Its pid is in {RUNS / tag / gates.RUNNING_FILE}. Stop it, then re-run the Launch cell and this cell"
                )
    if all(states[tag] == "done" or tag in DEAD for tag in RUN_TAGS):
        break
    if not WAIT_FOR_RUNS:
        break
    start_due()
    time.sleep(POLL_SECONDS)

if DEAD:
    raise RuntimeError(
        f"{sorted(DEAD)} died (log tails above). Fix the cause, then re-run the Launch cell and this cell; "
        "a run with a last.pt resumes from it"
    )
if short_tags():
    raise RuntimeError(
        f"finished with fewer than {FULL_EPOCHS} epochs: {short_tags()} (tag: epochs run) and EXTEND is off. "
        f"Set EXTEND = True to train such a tag on to {FULL_EPOCHS}, or lower FULL_EPOCHS; then re-run the Launch cell and this cell."
    )

## Table

One row per tag, the run tags first and then the reference tags whose folders are in `RUNS`: best val FG mIoU and its epoch, the epochs run, the mean val FG mIoU of the last 10 epochs, the median seconds per epoch (epoch 1 left out), and the parameter count. `log` is `ok` when the run's `train.log` shows the expected images, resolution and model. `flags` is `ok` when the flags stored in the run's folder are this notebook's flags for the tag (see Launch), and `DIFFER` when they are not: such a row is not from these settings. Then the per-class val IoU at the best epoch.

**Still rising?** For each tag: the mean val FG mIoU of the last 10 epochs against the 10 before. More than 1 point up is printed as `still rising`. That is a rule of thumb to help decide whether to raise `FULL_EPOCHS` and extend; it is not a test.

**Skip minus no skip**, for each model, in points (1 point = 0.01 IoU), two ways: the best epoch of each run, and the mean of the last 10 epochs of each run. The single best epoch is noisy. The two runs must have the same number of epochs. If they do not, a warning line is printed and the comparison uses only the epochs both runs have. The no-skip run is notebook 09's, read from its folder; it is not trained again here. These are two separate single-seed runs, and more than the skip differs: the head's last two layers start from different random weights, the no-skip run is from an earlier session (commit 318358e), and run-to-run spread is not measured. Both no-skip runs were still rising at epoch 60. A difference of a few points in FG mIoU is not evidence of an effect; a thin class moving up from near zero is the thing to look for.

**Fused minus control, with the skip**, the same two ways. If both reference folders are there, the same lines without the skip follow. The skip is the same on both models, so the line still compares the two encoders. But with the skip both heads read each pixel's 224 bands directly, as the control's tokenizer does, so on a class where the probe's tokens-zeroed row is high this line says little about band tokens. Do not read the change in the gap between the two lines as an effect: it is four runs, one seed each.

**Skip probe.** For each finished run tag, `python -m sw.skip_probe` scores its `best.pt` on the validation split three ways: as trained, with the skip zeroed (transformer path only), and with the token path zeroed (skip path only). The first row must reproduce the logged score. Zeroing a path of a jointly trained model is an ablation at test time, not a retrained model: a row with one path off can read lower or higher than the first row. A low tokens-zeroed row does not show that the transformer is needed. Do not compare the skip-zeroed row with a reference row. A probe that fails is printed and does not stop the table. `PROBE = False` leaves it out.

In [ ]:
from sw.data import CLASS_NAMES

LAST = 10  # epochs in the "last epochs" mean
RISING_POINTS = 1.0  # last 10 against the 10 before: more than this is printed as "still rising"


def cell(value):
    return "  n/a" if value is None else f"{value:.3f}"


def mean(values):
    kept = [v for v in values if v is not None]
    return sum(kept) / len(kept) if kept else None


def points(a, b):
    """a minus b in points (1 point = 0.01 IoU), as text."""
    return "   n/a" if a is None or b is None else f"{100 * (a - b):+6.1f}"


def summarize(rows):
    """Best epoch, and means over the last LAST epochs, of a list of metrics rows."""
    best = max(rows, key=lambda r: -1.0 if r.get("val_fg_miou") is None else r["val_fg_miou"])
    late = rows[-LAST:]
    before = rows[-2 * LAST:-LAST] if len(rows) >= 2 * LAST else []
    return {
        "best_val_fg_miou": best.get("val_fg_miou"),
        "best_epoch": best["epoch"],
        "epochs": rows[-1]["epoch"],
        "per_class_iou_at_best": best.get("per_class_iou") or {},
        "last_epochs": len(late),
        "last_mean": mean(r.get("val_fg_miou") for r in late),
        "before_mean": mean(r.get("val_fg_miou") for r in before),
        "per_class_last_mean": {
            name: mean((r.get("per_class_iou") or {}).get(name) for r in late) for name in CLASS_NAMES
        },
    }


print(
    f"{'tag':22s} {'state':8s} {'best val FG':>11s} {'epoch':>5s} {'epochs':>6s} {'last-10 mean':>12s} "
    f"{'s/epoch':>8s} {'parameters':>10s}  {'log':9s} flags"
)
ROWS, TABLE = {}, {}
REFERENCE_FOUND = [tag for tag in REFERENCE_TAGS if (RUNS / tag).is_dir()]
for tag in [*RUN_TAGS, *REFERENCE_FOUND]:
    out = RUNS / tag
    state = gates.run_state(out)
    complete, wrong = log_check(tag)
    log = "WRONG" if wrong else "ok" if complete else "not found"
    flags = "DIFFER" if flags_differ(tag) else "ok" if (out / gates.ARGS_FILE).is_file() else "no record"
    try:
        ROWS[tag] = gates.read_metrics(out / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        print(f"{tag:22s} {state:8s} no epoch finished yet")
        continue
    params = None
    if (out / gates.DONE_FILE).is_file():
        params = json.loads((out / gates.DONE_FILE).read_text())["result"].get("param_count")
    TABLE[tag] = row = {
        **summarize(ROWS[tag]),
        "state": state,
        "median_epoch_seconds": gates.median_epoch_seconds(ROWS[tag]),
        "param_count": params,
        "log": log,
        "flags": flags,
    }
    sec = row["median_epoch_seconds"]
    print(
        f"{tag:22s} {state:8s} {cell(row['best_val_fg_miou']):>11s} {row['best_epoch']:5d} {row['epochs']:6d} "
        f"{cell(row['last_mean']):>12s} {'n/a' if sec is None else format(sec, '.1f'):>8s} "
        f"{'n/a' if params is None else format(params, ','):>10s}  {log:9s} {flags}"
    )
for tag in REFERENCE_TAGS:
    if tag not in REFERENCE_FOUND:
        print(
            f"{tag}: no folder in {RUNS}. This reference (notebook 09, no skip) is not shown, "
            f"and there is no 'skip minus no skip' line for {SKIP_OF[tag]}."
        )

print("\nper-class val IoU at the best epoch")
print(f"{'tag':22s} " + " ".join(f"{name[:10]:>10s}" for name in CLASS_NAMES))
for tag, row in TABLE.items():
    print(f"{tag:22s} " + " ".join(f"{cell(row['per_class_iou_at_best'].get(name)):>10s}" for name in CLASS_NAMES))

print(f"\nstill rising? mean val FG mIoU of the last {LAST} epochs against the {LAST} before")
for tag, row in TABLE.items():
    if row["before_mean"] is None or row["last_mean"] is None:
        print(f"{tag:22s} n/a (needs {2 * LAST} epochs; has {row['epochs']})")
        continue
    gain = 100 * (row["last_mean"] - row["before_mean"])
    word = "still rising" if gain > RISING_POINTS else "not clearly rising"
    print(
        f"{tag:22s} {row['before_mean']:.3f} -> {row['last_mean']:.3f}  {gain:+.1f} points  {word} "
        f"(rule of thumb: more than {RISING_POINTS:.0f} point up)"
    )

unfinished = [tag for tag in [*RUN_TAGS, *REFERENCE_FOUND] if gates.run_state(RUNS / tag) != "done"]
if unfinished:
    print(f"\nnot finished: {unfinished}. Their rows are not final; re-run this cell when they are done.")

differ = [tag for tag, row in TABLE.items() if row["flags"] == "DIFFER"]
if differ:
    print(f"\nWARNING: {differ} were started with flags that differ from this notebook's. Their rows are not from these settings.")
wrong_log = [tag for tag, row in TABLE.items() if row["log"] == "WRONG"]
if wrong_log:
    print(
        f"\nWARNING: the train.log of {wrong_log} shows another model, resolution or image count than this notebook "
        "expects for the tag. Do not read the comparison lines that use them."
    )

COMPARISONS = {}


def compare(key, title, a_tag, b_tag, a_name, b_name):
    """Print a minus b in points, over the epochs both runs have. Nothing is compared if a run has no rows."""
    print(f"\n{title}, in points (1 point = 0.01 IoU)")
    missing = [tag for tag in (a_tag, b_tag) if tag not in ROWS]
    if missing:
        print(f"needs both {a_tag} and {b_tag}; not compared (no rows for {', '.join(missing)}).")
        return
    n_a, n_b = TABLE[a_tag]["epochs"], TABLE[b_tag]["epochs"]
    common = min(n_a, n_b)
    print(f"epochs run: {a_tag} {n_a}, {b_tag} {n_b}")
    if n_a != n_b:
        print(
            f"WARNING: the two runs have different epoch counts ({n_a} and {n_b}). "
            f"The comparison below uses epochs 1 to {common} of each."
        )
    a = summarize([r for r in ROWS[a_tag] if r["epoch"] <= common])
    b = summarize([r for r in ROWS[b_tag] if r["epoch"] <= common])
    COMPARISONS[key] = {"epochs": common, a_name: a, b_name: b}
    print(f"{'':48s} {'FG mIoU':>7s} " + " ".join(f"{name[:10]:>10s}" for name in CLASS_NAMES))
    label = f"best epoch of each ({a_name} {a['best_epoch']}, {b_name} {b['best_epoch']})"
    print(
        f"{label:48s} {points(a['best_val_fg_miou'], b['best_val_fg_miou']):>7s} "
        + " ".join(
            f"{points(a['per_class_iou_at_best'].get(name), b['per_class_iou_at_best'].get(name)):>10s}"
            for name in CLASS_NAMES
        )
    )
    label = f"mean of the last {a['last_epochs']} epochs of each"
    print(
        f"{label:48s} {points(a['last_mean'], b['last_mean']):>7s} "
        + " ".join(f"{points(a['per_class_last_mean'][name], b['per_class_last_mean'][name]):>10s}" for name in CLASS_NAMES)
    )


for tag in REFERENCE_FOUND:
    compare(f"skip:{tag}", f"skip minus no skip: {SKIP_OF[tag]} minus {tag}", SKIP_OF[tag], tag, "skip", "no skip")
compare("with skip", f"fused minus control, with the skip: {FUSED} minus {CONTROL}", FUSED, CONTROL, "fused", "control")
if len(REFERENCE_FOUND) == 2:
    compare("no skip", "fused minus control, without the skip (notebook 09's runs)", *REFERENCE_TAGS, "fused", "control")
print("\nThe single best epoch is noisy; read each last-epochs row with it.")

print("\nskip probe: best.pt of each finished run tag on the validation split, three ways")
if not PROBE:
    print("skipped (PROBE is off)")
for tag in RUN_TAGS if PROBE else []:
    try:
        best = RUNS / tag / "best.pt"
        if gates.run_state(RUNS / tag) != "done" or not best.is_file():
            print(f"--- {tag}: not finished, or no best.pt; not probed")
            continue
        probe = subprocess.run(
            [
                sys.executable, "-m", "sw.skip_probe", "--checkpoint", str(best), "--data-root", str(SW_ROOT),
                "--device", DEVICE, "--num-workers", str(NUM_WORKERS),
            ],
            capture_output=True, text=True, env=ENV, cwd=REPO, timeout=PROBE_TIMEOUT,
        )
        if probe.returncode == 0:
            print(f"--- {tag}\n{probe.stdout.rstrip()}")
        else:
            why = (probe.stderr.strip().splitlines() or [f"exit code {probe.returncode}"])[-1]
            print(f"--- {tag}: the probe failed; the table above does not depend on it. {why}")
    except Exception as err:  # the probe is an extra: it must not stop the table
        print(f"--- {tag}: the probe could not run; the table above does not depend on it. {type(err).__name__}: {err}")

ran = ", ".join(f"{tag} {row['epochs']}" for tag, row in TABLE.items()) or "none"
print(
    f"\nOne seed, validation only, constant learning rate. Epochs run (of {FULL_EPOCHS} asked): {ran}. "
    "The protocol in docs/RECIPE.md is up to 200 epochs, three seeds, and a warmup-then-cosine schedule. "
    "This is a screen, not the protocol run, and it does not settle the hypothesis.\n"
    "The head skip was added after notebook 09's validation numbers showed the thin classes near zero. "
    "It was chosen on this validation set, so a gain here is not an independent test of it.\n"
    "The skip is a per-pixel path that does not pass through the transformer. In the probe, a gain on video_tape "
    "that stays in the 'tokens zeroed (skip path only)' row means the skip alone does it. "
    "The probe rows are an ablation at test time, not retrained models. A low 'tokens zeroed' row does not show "
    "that the transformer is needed. Do not compare the 'skip zeroed' row with a run trained without the skip.\n"
    "The skip is the same on both models, so fused minus control still compares the two encoders. But with the skip "
    "both heads read each pixel's 224 bands directly, as the control's tokenizer does. On a class where the "
    "'tokens zeroed' row is high, fused minus control says little about band tokens. Do not read the change in the gap "
    "between the with-skip and without-skip lines as an effect: it is four runs, one seed each.\n"
    + (
        "The reference rows are notebook 09's runs, read from their folders and not trained again here. "
        "Skip minus no skip compares two separate single-seed runs: the head's last two layers start from different "
        "random weights, the no-skip run is from an earlier session (notebook 09 ran at commit 318358e), and run-to-run "
        "spread is not measured. In notebook 09 both no-skip runs were still rising at epoch 60 (docs/RESULTS.md; the "
        "still-rising lines above show these folders). A difference of a few points in FG mIoU is not "
        "evidence of an effect; a thin class moving up from near zero is the thing to look for. "
        if REFERENCE_FOUND else
        "No reference folder was found, so nothing above compares a run with the skip against one without it. "
    )
    + "Report each difference in points as printed. "
    'Do not use a stronger word than "competitive" for any comparison, and only with the numbers beside it.'
)
print(f"commit {COMMIT}  runs {RUNS}")
print(f"notebook ran in {(time.time() - T_START) / 60:.1f} min")

## What to send back

* The table above with the probe rows, and `RUNS/<tag>/metrics.jsonl` and `RUNS/<tag>/DONE.json` for each run tag.
* `docs/RESULTS.md` gets one row per run tag, with its flags (`--head-skip` among them), the epochs run, and the words *one seed, validation only* in the notes. The probe rows go in the notes, as an ablation at test time.

To train longer, raise `FULL_EPOCHS` and Run all again. Both run tags are then resumed to the new count, one after the other. The reference tags stay as notebook 09 left them, so the comparison then uses only the epochs both runs have. That is a longer screen. It is not the protocol run.

Nothing here touches the test split.